In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import affine_transform, binary_erosion, generate_binary_structure

def analyze_tooth_ct(tooth_mask_path: str, ct_data, z_spacing: float, shell_vox: int = 2):
    """
    Phân tích dữ liệu CT của răng và tạo biểu đồ hồ sơ HU cùng với hình chiếu 2D.
    """
    # =========================================================
    # 2. TẢI DỮ LIỆU
    # =========================================================
    mask_img = nib.load(tooth_mask_path)
    mask_data = mask_img.get_fdata()
    mask = (mask_data > 0).astype(np.uint8)

    if mask.shape != ct_data.shape:
        raise ValueError(f"Mask shape {mask.shape} khác CT shape {ct_data.shape}")

    coords = np.argwhere(mask > 0)
    if len(coords) == 0:
        raise ValueError("Tooth mask rỗng.")

    # =========================================================
    # 3. GIỮ LẠI LỚP VOXEL NGOÀI CÙNG (OUTER SHELL)
    # =========================================================
    structure = generate_binary_structure(rank=3, connectivity=1)
    mask_eroded = binary_erosion(mask, structure=structure, iterations=shell_vox)
    outer_shell = mask.astype(bool) & (~mask_eroded)

    tooth_shell_hu = np.zeros_like(ct_data, dtype=np.float32)
    tooth_shell_hu[outer_shell] = ct_data[outer_shell]

    # =========================================================
    # 4. PCA TRÊN TOOTH MASK ĐỂ XOAY TRỤC DÀI VỀ Z
    # =========================================================
    centroid = coords.mean(axis=0)
    coords_centered = coords - centroid

    cov = np.cov(coords_centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)

    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]

    principal_axis = eigvecs[:, 0]
    second_axis = eigvecs[:, 1]
    third_axis = eigvecs[:, 2]

    if np.dot(principal_axis, [0, 0, 1]) < 0:
        principal_axis = -principal_axis

    R = np.stack([second_axis, third_axis, principal_axis], axis=0)
    offset = centroid - R.T @ centroid

    rot_shell_hu = affine_transform(
        tooth_shell_hu,
        matrix=R.T,
        offset=offset,
        order=1,
        mode='constant',
        cval=0.0
    )

    rot_shell_mask = affine_transform(
        outer_shell.astype(np.float32),
        matrix=R.T,
        offset=offset,
        order=0,
        mode='constant',
        cval=0.0
    )
    rot_shell_mask = rot_shell_mask > 0.5

    # =========================================================
    # 5. TÍNH MEAN HU THEO TỪNG LÁT Z
    # =========================================================
    mean_hu_list, n_voxel_list, slice_idx_list, z_mm_list = [], [], [], []
    std_hu_list, min_hu_list, max_hu_list = [], [], []

    for z in range(rot_shell_hu.shape[2]):
        shell_slice_mask = rot_shell_mask[:, :, z]
        values = rot_shell_hu[:, :, z][shell_slice_mask]

        if values.size > 0:
            mean_hu, std_hu = values.mean(), values.std()
            min_hu, max_hu = values.min(), values.max()
            n_vox = values.size
        else:
            mean_hu = std_hu = min_hu = max_hu = np.nan
            n_vox = 0

        slice_idx_list.append(z)
        z_mm_list.append(z * z_spacing)
        mean_hu_list.append(mean_hu)
        n_voxel_list.append(n_vox)
        std_hu_list.append(std_hu)
        min_hu_list.append(min_hu)
        max_hu_list.append(max_hu)

    df = pd.DataFrame({
        "slice_index": slice_idx_list, "z_mm": z_mm_list, "n_shell_voxels": n_voxel_list,
        "mean_HU": mean_hu_list, "std_HU": std_hu_list, "min_HU": min_hu_list, "max_HU": max_hu_list
    })

    df_valid = df[df["n_shell_voxels"] > 0].reset_index(drop=True)

    # =========================================================
    # Tạo hình chiếu XZ & Vẽ biểu đồ
    # =========================================================
    proj_on_y_full_z = rot_shell_mask.max(axis=1).astype(np.uint8)
    col_mask = proj_on_y_full_z.any(axis=0)
    col_indices_cropped = np.where(col_mask)[0]

    if len(col_indices_cropped) > 0:
        x_start_slice_index, x_end_slice_index = col_indices_cropped.min(), col_indices_cropped.max()
        proj_on_y_final = proj_on_y_full_z[:, x_start_slice_index : x_end_slice_index + 1]
    else:
        proj_on_y_final, x_start_slice_index, x_end_slice_index = np.array([]), 0, 0

    if proj_on_y_final.size > 0:
        row_mask = proj_on_y_final.any(axis=1)
        row_indices_cropped = np.where(row_mask)[0]
        if len(row_indices_cropped) > 0:
            r0_idx_final, r1_idx_final = row_indices_cropped.min(), row_indices_cropped.max()
            proj_on_y_final = proj_on_y_final[r0_idx_final : r1_idx_final + 1, :]
        else:
            r0_idx_final, r1_idx_final = 0, 0

    if x_end_slice_index >= x_start_slice_index:
        x_for_plot_and_image = np.arange(x_start_slice_index, x_end_slice_index + 1)
    else:
        x_for_plot_and_image = np.array([])

    if proj_on_y_final.size > 0:
        y_height_img, img_width = proj_on_y_final.shape[0], x_end_slice_index - x_start_slice_index + 1
        data_aspect_ratio = y_height_img / img_width
    else:
        y_height_img, img_width, data_aspect_ratio = 0, 1, 1

    ax1_height = 4.0
    ax2_height = 10.0 * data_aspect_ratio
    fig_height = ax1_height + ax2_height + 1.5

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, fig_height), sharex=True, gridspec_kw={'height_ratios': [ax1_height, ax2_height]})

    if len(x_for_plot_and_image) > 0:
        df_plot = df_valid[df_valid["slice_index"].isin(x_for_plot_and_image)]
    else:
        df_plot = pd.DataFrame(columns=df_valid.columns)

    ax1.plot(df_plot["slice_index"], df_plot["mean_HU"], color="darkblue", linewidth=2, label="Mean HU")
    ax1.fill_between(df_plot["slice_index"], df_plot["mean_HU"] - df_plot["std_HU"], df_plot["mean_HU"] + df_plot["std_HU"], color='lightcoral', alpha=0.3, label='Mean +/- Std Dev')
    ax1.plot(df_plot["slice_index"], df_plot["min_HU"], color="green", linestyle=':', linewidth=1, label="Min HU")
    ax1.plot(df_plot["slice_index"], df_plot["max_HU"], color="purple", linestyle='-.', linewidth=1, label="Max HU")

    ax1.set_ylabel("HU values")
    ax1.set_title(f"{tooth_mask_path.split('/')[-1]}")
    ax1.grid(True, alpha=0.3)

    if len(x_for_plot_and_image) > 0:
        ax1.set_xlim(x_start_slice_index - 0.5, x_end_slice_index + 0.5)
    ax1.legend()

    if proj_on_y_final.size > 0 and len(x_for_plot_and_image) > 0:
        ax2.imshow(proj_on_y_final, cmap="gray", origin='lower',
                   extent=[x_start_slice_index - 0.5, x_end_slice_index + 0.5, 0, y_height_img], aspect='auto')

    ax2.set_xlabel("slice_index")
    ax2.set_ylabel("Y-dimension (rotated)")

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.05)

    # Note: Xóa dòng plt.savefig và plt.show ở đây để tránh lưu rác và đỡ chậm notebook.
    return fig, df_valid


In [ ]:
import os
import matplotlib.pyplot as plt

def cej_analyze_tooth_ct(ct_data, z_spacing, n_shell_voxels: int):
  output_dir = os.path.join('/content/drive/MyDrive/CEJ/P2', str(n_shell_voxels),)
  os.makedirs(output_dir, exist_ok=True)

  tooth_path = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person2/'
  teeth = [
      'upper_right_central_incisor_fdi11.nii.gz', 'upper_right_canine_fdi13.nii.gz',
      'upper_right_first_premolar_fdi14.nii.gz', 'upper_right_first_molar_fdi16.nii.gz',
      'upper_left_central_incisor_fdi21.nii.gz', 'upper_left_canine_fdi23.nii.gz',
      'upper_left_first_premolar_fdi24.nii.gz', 'upper_left_first_molar_fdi26.nii.gz',
      'lower_right_central_incisor_fdi41.nii.gz', 'lower_right_canine_fdi43.nii.gz',
      'lower_right_first_premolar_fdi44.nii.gz', 'lower_right_first_molar_fdi46.nii.gz',
      'lower_left_central_incisor_fdi31.nii.gz', 'lower_left_canine_fdi33.nii.gz',
      'lower_left_first_premolar_fdi34.nii.gz', 'lower_left_first_molar_fdi36.nii.gz'
  ]

  for tooth in teeth:
    print(f"Start {tooth}")
    tooth_mask_path = os.path.join(tooth_path, tooth)
    fig_output, df_output = analyze_tooth_ct(tooth_mask_path, ct_data, z_spacing, n_shell_voxels)

    tooth_name_without_ext = tooth.split('.')[0]
    png_filepath = os.path.join(output_dir, tooth_name_without_ext + '.png')
    csv_filepath = os.path.join(output_dir, tooth_name_without_ext + '.csv')

    fig_output.savefig(png_filepath, dpi=300, bbox_inches="tight")
    df_output.to_csv(csv_filepath, index=False)

    # GIẢI PHÓNG BỘ NHỚ sau khi lưu
    plt.close(fig_output)
    print(f"Done {tooth}")


In [ ]:
import nibabel as nib

# n_voxel = [1, 2, 3, 4]
ct_path = '/content/drive/MyDrive/Pipeline/Data/Person2.nii.gz'

print("Đang tải CT Data...")
ct_img = nib.load(ct_path)
ct_data = ct_img.get_fdata()
# Lấy z_spacing một lần để truyền vào hàm
z_spacing = ct_img.header.get_zooms()[2]

# for i in n_voxel:
#   print(f"\n--- Shell voxel: {i} ---")
#   cej_analyze_tooth_ct(ct_data, z_spacing, i)
#   print(f"--- Done shell voxel: {i} ---")


In [ ]:
cej_analyze_tooth_ct(ct_data, z_spacing, 5)